![DBAcademy](./Includes/images/common/db-academy.png)

# Bonus Lab: Using AI Functions

### Duration: ~20 minutes

## Overview

This is a **self-paced lab** for you to practice Lakeflow Designer outside of the live workshop. 

It builds one more gold table on top of what you already built, this time showcasing the **AI Function** operator with `ai_sentiment`.

You will build a new gold table that answers: *"How did attendees feel about each free-form aspect of the session?"*

The final table has one row per free-form response with:
- **meeting_uuid**
- **session_name** (from `sessions_silver`)
- **answer** (the original free-form text)
- **question_id** and **question**
- **respondent_id**
- **sentiment** (positive, negative, neutral, or mixed) generated by `ai_sentiment`

## Learning Objectives

By the end of this lab, you will be able to:

- **Use** **Genie Code** (AI) to generate a Filter + AI Function + Output chain from a natural language prompt.
- **Apply** the **AI Function** operator with `ai_sentiment` to derive sentiment labels from free-form text feedback.
- **Join** the result with a silver dimension table to add descriptive metadata (`session_name`).
- **Inspect** AI-generated nodes and verify the output before trusting it.

## REQUIRED PREREQUISITES
<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">PREREQUISITES</strong>
  <div style="color:#333;">
You must complete <strong>all of the previous demonstrations</strong> before starting this lab.

  </div>
</div>

## A. Apply `ai_sentiment` with Genie Code

In this step you will use **Genie Code** to generate a chain of nodes that filters **feedback_silver** to free-form responses and applies the `ai_sentiment` AI function. 

This is the AI-first authoring pattern: describe the result in natural language and let Designer compose the operators.

### The free-form questions

Three `question_id` values in **feedback_silver** are free-form text (not numeric ratings):

| question_id | Question |
|---|---|
| `r6Wd31x8THycYxzVzeZ0BA` | What did you like about the content in this session? |
| `-TYqDeIsREmunUvAmNrpPw` | What could have been improved? |
| `is7w6QM0RtK1Y3RsL_o8Vg` | What additional topics would you like a deep dive on? |

### What `ai_sentiment` returns

`ai_sentiment` is a built-in Databricks AI function. It takes a text string and returns one of four labels: `positive`, `negative`, `neutral`, or `mixed`. No model deployment required.

**Available AI Functions**: [AWS](https://docs.databricks.com/aws/en/designer/built-in-operators#ai-function) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/designer/built-in-operators#ai-function) | [GCP](https://docs.databricks.com/gcp/en/designer/built-in-operators#ai-function)

### A1. Create a New Visual Data Prep File in Lakeflow Designer

> Keeping this notebook on one tab and Lakeflow Designer on another makes the rest of the demo easier to follow.

1. In the **workspace sidebar**, select the **Folder** icon.

2. In your main folder, select **More Options** > **Create** > **Visual data prep**.
   - **NOTE:** You can also use the main navigation bar and select **+ New** > **Visual data prep**.

3. A new file opens with an auto-generated name like `Visual data prep YYYY-MM-DD HH:MM:SS`.

4. Click the name at the top of the page and rename the file to `meeting_gold_feedback_sentiment - YOUR INITIALS`.

5. **Lakeflow Designer** should now be open and ready to use.



### A2. Add the Source `feedback_silver_table`

1. From your **learn_no_code_etl_designer_USERNAME.designer_meeting** schema, add the **feedback_silver** table to the canvas.

#### Checkpoint - Add Source Silver Table
![Silver](./Includes/images/ai_visual_etl_pipeline/checkpoint-add-source-silver-table.png)

### A3. Filter and Transform the Table
1. Select the **feedback_silver** node.

2. In the **Genie Code** prompt bar, enter the following prompt:

<div style="background: #F9F7F4; border: 1px solid #EEEDE9; border-radius: 6px; margin: 8px 0; font-family: sans-serif; max-width: 1100px; overflow: hidden;">
  <div style="display: flex; justify-content: space-between; align-items: center; background: #EEEDE9; padding: 6px 12px; border-bottom: 1px solid #d9d7d2;">
    <span style="font-size: 10pt; font-weight: 700; color: #0b2026; letter-spacing: 1px; text-transform: uppercase;">Genie Code Prompt</span>
    <button
      type="button"
      aria-label="Copy Genie Code prompt to clipboard"
      onmouseover="this.style.background='#143F4D';"
      onmouseout="this.style.background='#1B5162';"
      onclick="
        var text = this.parentElement.nextElementSibling.innerText;
        var btn = this;
        var ok = false;
        try {
          var ta = document.createElement('textarea');
          ta.value = text;
          ta.style.position = 'fixed';
          ta.style.top = '0';
          ta.style.left = '0';
          ta.style.opacity = '0';
          document.body.appendChild(ta);
          ta.focus();
          ta.select();
          ok = document.execCommand('copy');
          document.body.removeChild(ta);
        } catch (e) { ok = false; }
        if (!ok && navigator.clipboard && navigator.clipboard.writeText) {
          navigator.clipboard.writeText(text).then(
            function(){ btn.innerText = '✓ Copied!'; setTimeout(function(){ btn.innerText = 'Copy'; }, 2000); },
            function(){ btn.innerText = 'Press Ctrl+C'; setTimeout(function(){ btn.innerText = 'Copy'; }, 2000); }
          );
        } else {
          btn.innerText = ok ? '✓ Copied!' : 'Press Ctrl+C';
          setTimeout(function(){ btn.innerText = 'Copy'; }, 2000);
        }
      "
      style="background: #1B5162; color: #FFFFFF; border: none; border-radius: 6px; padding: 6px 18px; font-size: 11pt; font-weight: 700; cursor: pointer; white-space: nowrap; transition: background 0.15s ease;">Copy</button>
  </div>
  <pre style="margin: 0; padding: 16px 18px; background: #FFFFFF; color: #0b2026; font-family: Menlo, Consolas, monospace; font-size: 12pt; line-height: 1.55; white-space: pre-wrap; word-wrap: break-word; overflow-x: auto;">
Using the feedback_silver table:
- Filter to only the three free-form question_ids: 'r6Wd31x8THycYxzVzeZ0BA', '-TYqDeIsREmunUvAmNrpPw', and 'is7w6QM0RtK1Y3RsL_o8Vg'.
- Filter out empty answer rows
- Add a new column called sentiment by applying the ai_sentiment AI function to the answer column.
- Keep all existing columns.
  </pre>
</div>



<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    Warning
  </strong>
  <div style="color:#333;">
AI Functions can take a few minutes to execute.
  </div>
</div>



3. **Genie Code** generates the nodes on the canvas. Review what was generated, then click **Accept all** if prompted.
    - **Filter** node
    - **add_sentiment** node (AI Function)

4. Select the generated AI Function node and view its **Preview** pane. Confirm:
    - Each row has a **sentiment** value of `positive`, `negative`, `neutral`, or `mixed`.
    - Only rows from the three free-form questions are present.
    - The original columns (**meeting_uuid**, **answer**, **question**, **question_id**, **respondent_id**) are still there.

5. If the result is not what you expected:
    - **Manually adjust** the generated node in its configuration panel.
    - **Re-prompt Genie Code** with a refined description.

6. Rename the generated AI Function node to `feedback_sentiment_transform`.

#### Checkpoint - Filter and AI Transform
![Silver](./Includes/images/ai_visual_etl_pipeline/checkpoint-ai-sentiment.png)

## B. TO DO: Join with Session Metadata

The output from the previous section includes **meeting_uuid** but not the human-readable **session_name** or other session details.

In this step you will **join** the sentiment result to **sessions_silver** on your own so the gold table is detailed and ready for analysis.

You can complete it:
- Manually with the **Join** and **Output** operators, or
- With **Genie Code** to build the entire join + output chain from one prompt.

### TO DO: Build the Following

1. Add the **sessions_silver** table to the canvas.

2. Add a **Join** operator between **sessions_silver_table** (left) and **feedback_sentiment_transform** (right).
    - Keep all columns from **sessions_silver_table**.
    - Keep only the **respondent_id**, **sentiment**, **answer**, and **question** columns from **feedback_sentiment_transform**.

3. Configure as a **Left join** on **meeting_uuid** so every sentiment row is preserved.

4. **Output** the result as a table named `gold_feedback_sentiment`.

5. Select **Run** on the final table.

6. Auto layout your canvas.

<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    Warning
  </strong>
  <div style="color:#333;">
AI Functions can take a few minutes to execute.
  </div>
</div>


#### Checkpoint - Join (Can Vary)
![Final Join](./Includes/images/ai_visual_etl_pipeline/checkpoint-gold-feedback-tbl.png)

## C. Validate Your Gold Table

The cell below uses a small Python helper (`check_table`) to validate your `gold_feedback_sentiment` table. It confirms two things:

- **Row count** matches what we expect (3,984 free-form responses, one row per response).
    - Recall the Lakeflow Designer preview pane only displays a sample of data.
- **Expected columns** are present.

If a check fails, the output prints a `[FAIL]` line. Row-count failures also print a hint about checking your JOIN configuration.

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Wait for the table to create</strong>
  <div style="color:#333;">

You must wait for Lakeflow Designer to create the final table, **gold_feedback_sentiment**, before proceeding to validation.

  </div>
</div>

### C1. REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless Compute, Version 5**

![Serverless Select](./Includes/images/common/select-serverless.png)
<br></br>
  - How to select an environment version:
[AWS](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version) |
[GCP](https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version)

**NOTE:** This notebook requires **SQL and Python, and was tested using Serverless V5**. All purpose compute may work but is not guaranteed to behave the same or support all features demonstrated.
  </div>
</div>


1. Run the cell to create the necessary validation function.

In [0]:
%run ./Includes/Classroom-Setup-Validation-Checker

ℹ️ Non-Vocareum workspace detected. Setting up catalog.
ℹ️ Using default catalog name: 'learn_no_code_etl_designer_puli_reddy'.
✅ Catalog 'learn_no_code_etl_designer_puli_reddy' already exists. Using this catalog.


DataFrame[]

### C2. Validate the `gold_feedback_sentiment` Table


In [0]:
check_table(
    table_name=f"{my_catalog}.{schema}.gold_feedback_sentiment",
    expected_rows=3984,
    expected_columns=[
        "event_date", "difficulty_level", "session_name", "meeting_uuid", "planned_duration_minutes",
        "presenter", "topic_category", "sentiment", "answer", "question"
    ]
)

Validating: learn_no_code_etl_designer_puli_reddy.designer_meeting.gold_feedback_sentiment
[PASS] Row count: actual=3984, expected=3984
[PASS] Columns present: all present
All checks passed.


True


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>